# Qwen Approximate Machine Unlearning

This notebook evaluates whether three approximate methods can move the frozen Original Qwen classifier towards the scenario-specific Full Retraining reference without rebuilding the baseline.

The experimental sequence is deliberate: Original Qwen, Full Retraining, Retain-Set Fine-Tuning, Gradient Ascent, Gradient Difference, then the overall comparison. The completed Original Qwen and Full Retraining experiments are read-only throughout.

In [214]:
# Keep every training switch disabled while the notebook is reviewed.
RUN_UNLEARNING = False
RUN_RETAIN_SET_FINE_TUNING = False
RUN_GRADIENT_ASCENT = False
RUN_GRADIENT_DIFFERENCE = False

BASELINE_RESULTS_READ_ONLY = True
FULL_RETRAINING_RESULTS_READ_ONLY = True

print("Revision safety state: all training switches are disabled.")

Revision safety state: all training switches are disabled.


In [215]:
from pathlib import Path
import gc
import json
import time

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, confusion_matrix,
    f1_score, log_loss, precision_score, recall_score, roc_auc_score,
)

In [216]:
def locate_final_submission():
    """Find code/final_submission without assuming the notebook working directory."""
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        direct = candidate if candidate.name == "final_submission" else candidate / "code" / "final_submission"
        if (direct / "data" / "final" / "kidney_transplant_assessments.csv").is_file():
            return direct.resolve()
    raise FileNotFoundError("Could not locate code/final_submission.")

FINAL_SUBMISSION = locate_final_submission()

def project_relative_path(path):
    return str(Path("code") / "final_submission" / Path(path).resolve().relative_to(FINAL_SUBMISSION))

In [217]:
# Final Original Qwen: run 20260829T151430Z, LR 1e-5, selected epoch 6, threshold 0.55.
# These canonical project artefacts are the only baseline model accepted here.
BASELINE_RUN_ID = "20260829T151430Z"
QWEN_MODEL_ROOT = FINAL_SUBMISSION / "models" / "qwen"
QWEN_RESULTS_ROOT = FINAL_SUBMISSION / "results" / "qwen"

BASELINE_MODEL_ROOT = QWEN_MODEL_ROOT / "baseline"
BASELINE_ADAPTER_DIR = BASELINE_MODEL_ROOT / "adapter"
BASELINE_HEAD_PATH = BASELINE_MODEL_ROOT / "binary_classification_head.pt"
BASELINE_RESULT_DIR = QWEN_RESULTS_ROOT / "original"

FULL_RETRAINING_MODEL_ROOT = QWEN_MODEL_ROOT / "full_retraining"
FULL_RETRAINING_ROOT = QWEN_RESULTS_ROOT / "full_retraining"
UNLEARNING_ROOT = QWEN_RESULTS_ROOT / "unlearning"
APPROXIMATE_MODEL_ROOTS = {
    "retain_set_fine_tuning": QWEN_MODEL_ROOT / "retain_set_fine_tuning",
    "gradient_ascent": QWEN_MODEL_ROOT / "gradient_ascent",
    "gradient_difference": QWEN_MODEL_ROOT / "gradient_difference",
}

assert BASELINE_RESULTS_READ_ONLY and FULL_RETRAINING_RESULTS_READ_ONLY
assert UNLEARNING_ROOT.resolve() not in {BASELINE_RESULT_DIR.resolve(), FULL_RETRAINING_ROOT.resolve()}
assert BASELINE_MODEL_ROOT.resolve() not in {path.resolve() for path in APPROXIMATE_MODEL_ROOTS.values()}
assert FULL_RETRAINING_MODEL_ROOT.resolve() not in {path.resolve() for path in APPROXIMATE_MODEL_ROOTS.values()}
display(pd.Series(
    {method: project_relative_path(path) for method, path in APPROXIMATE_MODEL_ROOTS.items()},
    name="Future model-output root",
).to_frame())

,Future model-output root
retain_set_fine_tuning,code/final_submission/models/qwen/retain_set_f...
gradient_ascent,code/final_submission/models/qwen/gradient_ascent
gradient_difference,code/final_submission/models/qwen/gradient_dif...


# 1. Original Qwen Baseline

The Original Qwen classifier represents the model before any deletion request is applied. It has already been trained. This section loads frozen saved evidence only; it contains no baseline training or threshold selection.

## 1.1 Load Frozen Baseline

The sole authoritative baseline is run `20260829T151430Z`: learning rate `1e-5`, selected epoch `6`, and frozen threshold `0.55`. Its frozen selected model is stored under `code/final_submission/models/qwen/baseline/`. No legacy RunPod or experimental path is searched.

In [218]:
baseline_model_locations = pd.Series({
    "Baseline Model Root": project_relative_path(BASELINE_MODEL_ROOT),
    "Baseline Adapter": project_relative_path(BASELINE_ADAPTER_DIR),
    "Baseline Classification Head": project_relative_path(BASELINE_HEAD_PATH),
}, name="Canonical read-only path").to_frame()
display(baseline_model_locations)

,Canonical read-only path
Baseline Model Root,code/final_submission/models/qwen/baseline
Baseline Adapter,code/final_submission/models/qwen/baseline/ada...
Baseline Classification Head,code/final_submission/models/qwen/baseline/bin...


In [219]:
baseline_paths = {
    "Adapter configuration": BASELINE_ADAPTER_DIR / "adapter_config.json",
    "Adapter weights": BASELINE_ADAPTER_DIR / "adapter_model.safetensors",
    "Processor configuration": BASELINE_ADAPTER_DIR / "processor_config.json",
    "Tokenizer": BASELINE_ADAPTER_DIR / "tokenizer.json",
    "Tokenizer configuration": BASELINE_ADAPTER_DIR / "tokenizer_config.json",
    "Classification head": BASELINE_HEAD_PATH,
    "Experiment configuration": BASELINE_RESULT_DIR / "experiment_configuration.json",
    "Frozen threshold": BASELINE_RESULT_DIR / "selected_threshold.json",
    "Serialisation specification": BASELINE_RESULT_DIR / "serialisation_specification.json",
    "Test metrics": BASELINE_RESULT_DIR / "retained_test_metrics.csv",
    "Test probabilities": BASELINE_RESULT_DIR / "retained_test_probabilities.csv",
}
baseline_path_table = pd.DataFrame([
    {"Artefact": name, "Path": project_relative_path(path), "Exists": path.is_file()}
    for name, path in baseline_paths.items()
])
display(baseline_path_table)

,Artefact,Path,Exists
0,Adapter configuration,code/final_submission/models/qwen/baseline/ada...,True
1,Adapter weights,code/final_submission/models/qwen/baseline/ada...,True
2,Processor configuration,code/final_submission/models/qwen/baseline/ada...,True
3,Tokenizer,code/final_submission/models/qwen/baseline/ada...,True
4,Tokenizer configuration,code/final_submission/models/qwen/baseline/ada...,True
5,Classification head,code/final_submission/models/qwen/baseline/bin...,True
6,Experiment configuration,code/final_submission/results/qwen/original/ex...,True
7,Frozen threshold,code/final_submission/results/qwen/original/se...,True
8,Serialisation specification,code/final_submission/results/qwen/original/se...,True
9,Test metrics,code/final_submission/results/qwen/original/re...,True


In [220]:
METHOD_SWITCH_ENABLED = any([
    RUN_RETAIN_SET_FINE_TUNING,
    RUN_GRADIENT_ASCENT,
    RUN_GRADIENT_DIFFERENCE,
])
GPU_EXECUTION_REQUESTED = RUN_UNLEARNING and METHOD_SWITCH_ENABLED
if METHOD_SWITCH_ENABLED and not RUN_UNLEARNING:
    print("Overall RUN_UNLEARNING safety gate is still disabled.")
required_baseline_model_files = [
    baseline_paths["Adapter configuration"], baseline_paths["Adapter weights"],
    baseline_paths["Processor configuration"], baseline_paths["Tokenizer"],
    baseline_paths["Tokenizer configuration"], baseline_paths["Classification head"],
]
missing_baseline_model = [path for path in required_baseline_model_files if not path.is_file()]
missing_verification_evidence = not baseline_paths["Test probabilities"].is_file()

if GPU_EXECUTION_REQUESTED and missing_baseline_model:
    raise FileNotFoundError(
        "The final frozen Original Qwen model bundle is incomplete. "
        "Expected files under code/final_submission/models/qwen/baseline/. "
        "Do not retrain the baseline. Missing: "
        + "; ".join(str(path) for path in missing_baseline_model)
    )
if GPU_EXECUTION_REQUESTED and missing_verification_evidence:
    raise FileNotFoundError(
        "Frozen Original Qwen predictions are required to verify reconstruction before training. "
        "Restore results/qwen/original/retained_test_probabilities.csv; do not retrain the baseline."
    )

if missing_baseline_model or missing_verification_evidence:
    print("LOCAL REVIEW MODE: saved results remain available; GPU reconstruction is blocked until all required evidence exists.")
else:
    print("Canonical baseline bundle preflight passed. No model has been loaded or trained.")

Canonical baseline bundle preflight passed. No model has been loaded or trained.


## 1.2 Baseline Configuration

In [221]:
baseline_metrics = pd.read_csv(baseline_paths["Test metrics"])
assert len(baseline_metrics) == 1

baseline_config = (
    json.loads(baseline_paths["Experiment configuration"].read_text())
    if baseline_paths["Experiment configuration"].is_file() else None
)
threshold_record = (
    json.loads(baseline_paths["Frozen threshold"].read_text())
    if baseline_paths["Frozen threshold"].is_file() else None
)
serialisation_spec = (
    json.loads(baseline_paths["Serialisation specification"].read_text())
    if baseline_paths["Serialisation specification"].is_file() else None
)
baseline_probabilities = (
    pd.read_csv(baseline_paths["Test probabilities"])
    if baseline_paths["Test probabilities"].is_file() else pd.DataFrame()
)

In [222]:
if baseline_config is not None:
    training = baseline_config["training"]
    baseline_configuration = {
        "Run ID": baseline_config["run_id"],
        "Model": baseline_config["model_id"],
        "Learning Rate": training["learning rate"],
        "Selected Epoch": baseline_config["best_epoch"],
        "Batch Size": training["batch size"],
        "Gradient Accumulation": training.get("effective batch size", 32) // training["batch size"],
        "Effective Batch Size": training.get("effective batch size", 32),
        "LoRA Rank": training["LoRA rank"],
        "LoRA Alpha": training["LoRA alpha"],
        "LoRA Dropout": training["LoRA dropout"],
        "Frozen Threshold": baseline_config["selected_threshold"],
    }
else:
    provenance = json.loads((BASELINE_RESULT_DIR / "PROVENANCE.json").read_text())
    baseline_configuration = {
        "Run ID": provenance["source_run_id"],
        "Model": "Qwen3.5-2B classifier", "Learning Rate": provenance["learning_rate"],
        "Selected Epoch": provenance["selected_epoch"], "Frozen Threshold": provenance["frozen_threshold"],
    }
display(pd.Series(baseline_configuration, name="Value").to_frame())

,Value
Run ID,20260829T151430Z
Model,unsloth/Qwen3.5-2B-Base
Learning Rate,0.00001
Selected Epoch,6
Batch Size,8
Gradient Accumulation,4
Effective Batch Size,32
LoRA Rank,16
LoRA Alpha,16
LoRA Dropout,0


In [223]:
assert baseline_config is not None and threshold_record is not None and serialisation_spec is not None
assert len(baseline_probabilities) == 8_988
assert baseline_probabilities["assessment_id"].is_unique
assert list(baseline_probabilities.columns) == [
    "assessment_id", "label", "logit_0", "logit_1", "probability_class_1", "prediction",
]
FROZEN_THRESHOLD = float(threshold_record["threshold"])
assert baseline_config["run_id"] == BASELINE_RUN_ID
assert baseline_config["model_id"] == "unsloth/Qwen3.5-2B-Base"
assert np.isclose(baseline_configuration["Learning Rate"], 1e-5)
assert baseline_configuration["Selected Epoch"] == 6
assert np.isclose(FROZEN_THRESHOLD, 0.55)
assert np.isclose(float(baseline_metrics.iloc[0]["threshold"]), FROZEN_THRESHOLD)
assert baseline_configuration["Batch Size"] == 8
assert baseline_configuration["Gradient Accumulation"] == 4
assert baseline_configuration["Effective Batch Size"] == 32
assert baseline_configuration["LoRA Rank"] == 16
assert baseline_configuration["LoRA Alpha"] == 16
assert baseline_configuration["LoRA Dropout"] == 0

adapter_configuration = json.loads(baseline_paths["Adapter configuration"].read_text())
MODEL_ID = adapter_configuration["base_model_name_or_path"]
MAX_SEQ_LENGTH = int(baseline_config["max_seq_length"]) if baseline_config is not None else 216

## 1.3 Overall Original Qwen Baseline

In [224]:
UTILITY_KEYS = [
    "pr_auc", "balanced_accuracy", "binary_cross_entropy", "f1",
    "auroc", "precision", "recall", "specificity",
]
UTILITY_NAMES = {
    "pr_auc": "PR-AUC", "balanced_accuracy": "Balanced Accuracy",
    "binary_cross_entropy": "BCE", "f1": "F1", "auroc": "AUROC",
    "precision": "Precision", "recall": "Recall", "specificity": "Specificity",
}

def compute_saved_utility(predictions):
    """Calculate utility from frozen saved probabilities at threshold 0.55."""
    y = predictions["label"].to_numpy()
    probability = predictions["probability_class_1"].to_numpy()
    predicted = (probability >= FROZEN_THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, predicted, labels=[0, 1]).ravel()
    return {
        "pr_auc": average_precision_score(y, probability),
        "balanced_accuracy": balanced_accuracy_score(y, predicted),
        "binary_cross_entropy": log_loss(y, probability, labels=[0, 1]),
        "f1": f1_score(y, predicted, zero_division=0),
        "auroc": roc_auc_score(y, probability),
        "precision": precision_score(y, predicted, zero_division=0),
        "recall": recall_score(y, predicted, zero_division=0),
        "specificity": tn / (tn + fp),
    }
baseline_utility = baseline_metrics[UTILITY_KEYS].rename(columns=UTILITY_NAMES)
display(baseline_utility.round(4))

,PR-AUC,Balanced Accuracy,BCE,F1,AUROC,Precision,Recall,Specificity
0,0.1762,0.6049,0.4323,0.2324,0.7287,0.1792,0.3308,0.8789


## 1.4 Baseline Result Summary

These saved values are the **Overall Original Qwen Baseline** on all 8,988 test rows. The same frozen model starts every deletion scenario; the scenario-matched table below evaluates its saved probabilities only on each scenario's retained-test rows.

# 2. Full Retraining Reference

Full Retraining rebuilds a model after the requested forget data has been removed. Its five scenario runs are already complete. This notebook loads their saved outputs as the scenario-specific reference and never calls Full Retraining code.

## 2.1 Load Completed Full Retraining Results

In [225]:
SCENARIOS = [
    "recipient_withdrawal", "donor_withdrawal", "invalid_consent",
    "hospital_removal", "retention_expiry",
]
SCENARIO_LABELS = {
    "recipient_withdrawal": "Recipient Withdrawal",
    "donor_withdrawal": "Donor Withdrawal",
    "invalid_consent": "Invalid Consent",
    "hospital_removal": "Hospital Removal",
    "retention_expiry": "Retention Expiry",
}
EXPECTED_FORGET_ROWS = {
    "recipient_withdrawal": 426, "donor_withdrawal": 1_992,
    "invalid_consent": 4_148, "hospital_removal": 4_314,
    "retention_expiry": 6_262,
}

In [226]:
# Full Retraining is loaded only as the reference; no model is trained here.
FULL_FILES = [
    "COMPLETE.json", "retained_test_metrics.csv",
    "retained_test_probabilities.csv", "forget_set_probabilities.csv",
    "training_history.csv",
]
full_retraining_model_paths = pd.DataFrame([
    {
        "Scenario": SCENARIO_LABELS[scenario],
        "Model Root": project_relative_path(FULL_RETRAINING_MODEL_ROOT / scenario),
        "Adapter": project_relative_path(FULL_RETRAINING_MODEL_ROOT / scenario / "adapter"),
        "Binary Head": project_relative_path(FULL_RETRAINING_MODEL_ROOT / scenario / "binary_classification_head.pt"),
        "Model Bundle Exists": (FULL_RETRAINING_MODEL_ROOT / scenario / "adapter" / "adapter_model.safetensors").is_file()
            and (FULL_RETRAINING_MODEL_ROOT / scenario / "binary_classification_head.pt").is_file(),
    }
    for scenario in SCENARIOS
])
display(full_retraining_model_paths)

full_retraining = {}

for scenario in SCENARIOS:
    folder = FULL_RETRAINING_ROOT / scenario
    paths = {name: folder / name for name in FULL_FILES}
    missing = [str(path) for path in paths.values() if not path.is_file()]
    if missing:
        raise FileNotFoundError(
            "Completed Full Retraining result missing. Do not rerun Full Retraining automatically. "
            + "; ".join(missing)
        )
    complete = json.loads(paths["COMPLETE.json"].read_text())
    assert complete["status"] == "complete"
    assert complete["training_forget_rows"] == EXPECTED_FORGET_ROWS[scenario]
    full_retraining[scenario] = {
        "complete": complete,
        "metrics": pd.read_csv(paths["retained_test_metrics.csv"]).iloc[0],
        "retained": pd.read_csv(paths["retained_test_probabilities.csv"]),
        "forget": pd.read_csv(paths["forget_set_probabilities.csv"]),
        "history": pd.read_csv(paths["training_history.csv"]),
    }

,Scenario,Model Root,Adapter,Binary Head,Model Bundle Exists
0,Recipient Withdrawal,code/final_submission/models/qwen/full_retrain...,code/final_submission/models/qwen/full_retrain...,code/final_submission/models/qwen/full_retrain...,True
1,Donor Withdrawal,code/final_submission/models/qwen/full_retrain...,code/final_submission/models/qwen/full_retrain...,code/final_submission/models/qwen/full_retrain...,True
2,Invalid Consent,code/final_submission/models/qwen/full_retrain...,code/final_submission/models/qwen/full_retrain...,code/final_submission/models/qwen/full_retrain...,True
3,Hospital Removal,code/final_submission/models/qwen/full_retrain...,code/final_submission/models/qwen/full_retrain...,code/final_submission/models/qwen/full_retrain...,True
4,Retention Expiry,code/final_submission/models/qwen/full_retrain...,code/final_submission/models/qwen/full_retrain...,code/final_submission/models/qwen/full_retrain...,True


## 2.2 Full Retraining Results

In [227]:
full_rows = []
for scenario in SCENARIOS:
    reference = full_retraining[scenario]
    full_rows.append({
        "Scenario": SCENARIO_LABELS[scenario],
        "Forget Rows": EXPECTED_FORGET_ROWS[scenario],
        **{UTILITY_NAMES[key]: reference["metrics"][key] for key in UTILITY_KEYS},
        "Training Time": reference["complete"]["training_seconds"],
    })
full_results_table = pd.DataFrame(full_rows)
display(full_results_table.style.format({
    **{name: "{:.4f}" for name in UTILITY_NAMES.values()},
    "Forget Rows": "{:,.0f}", "Training Time": "{:.1f}",
}))

,Scenario,Forget Rows,PR-AUC,Balanced Accuracy,BCE,F1,AUROC,Precision,Recall,Specificity,Training Time
0,Recipient Withdrawal,426,0.1833,0.5681,0.3532,0.2048,0.7332,0.2257,0.1875,0.9488,10496.7
1,Donor Withdrawal,"1,992",0.1753,0.5439,0.3190,0.1581,0.7286,0.2630,0.1131,0.9747,27832.9
2,Invalid Consent,"4,148",0.1725,0.5644,0.3494,0.1919,0.7360,0.1981,0.1860,0.9429,31044.2
3,Hospital Removal,"4,314",0.1833,0.5611,0.3542,0.1932,0.7395,0.2262,0.1686,0.9536,29258.4
4,Retention Expiry,"6,262",0.1615,0.5493,0.3450,0.1677,0.7352,0.2074,0.1408,0.9578,30202.6


## 2.3 Scenario-Matched Original Qwen Utility

For each deletion scenario, the frozen Original Qwen probabilities are restricted to the exact retained-test assessment IDs used by that Full Retraining reference. No Original Qwen inference is performed. Deltas then compare like-for-like retained-test rows.

In [228]:
scenario_original = {}
scenario_original_rows = []
for scenario in SCENARIOS:
    reference = full_retraining[scenario]
    retained_ids = reference["retained"][["assessment_id", "label"]].rename(columns={"label": "scenario_label"})
    assert retained_ids["assessment_id"].is_unique
    matched = retained_ids.merge(
        baseline_probabilities[["assessment_id", "label", "probability_class_1"]]
            .rename(columns={"label": "original_label"}),
        on="assessment_id", how="inner", validate="one_to_one",
    )
    expected_rows = int(reference["complete"]["retained_test_rows"])
    assert len(retained_ids) == expected_rows
    assert len(matched) == expected_rows
    assert matched["scenario_label"].equals(matched["original_label"])
    saved_predictions = matched[["assessment_id", "original_label", "probability_class_1"]].rename(
        columns={"original_label": "label"}
    )
    metrics = pd.Series(compute_saved_utility(saved_predictions))
    scenario_original[scenario] = {"metrics": metrics, "predictions": saved_predictions}
    scenario_original_rows.append({
        "Scenario": SCENARIO_LABELS[scenario], "Retained Test Rows": expected_rows,
        **{f"Original {UTILITY_NAMES[key]}": metrics[key] for key in UTILITY_KEYS},
    })

scenario_matched_original_utility = pd.DataFrame(scenario_original_rows)
display(scenario_matched_original_utility.style.format({
    "Retained Test Rows": "{:,.0f}",
    **{f"Original {UTILITY_NAMES[key]}": "{:.4f}" for key in UTILITY_KEYS},
}))

,Scenario,Retained Test Rows,Original PR-AUC,Original Balanced Accuracy,Original BCE,Original F1,Original AUROC,Original Precision,Original Recall,Original Specificity
0,Recipient Withdrawal,"8,898",0.1778,0.6059,0.4316,0.2335,0.7298,0.1800,0.3323,0.8795
1,Donor Withdrawal,"8,496",0.1761,0.6053,0.4330,0.2324,0.7277,0.1785,0.3328,0.8777
2,Invalid Consent,"8,078",0.1737,0.6037,0.4171,0.2264,0.7345,0.1739,0.3246,0.8829
3,Hospital Removal,"8,124",0.1781,0.6082,0.4326,0.2370,0.7342,0.1822,0.3388,0.8776
4,Retention Expiry,"7,614",0.1738,0.5945,0.3989,0.2257,0.7434,0.1852,0.2888,0.9003


In [229]:
delta_keys = ["pr_auc", "balanced_accuracy", "binary_cross_entropy", "f1", "auroc"]
full_delta_rows = []
for scenario in SCENARIOS:
    original = scenario_original[scenario]["metrics"]
    metrics = full_retraining[scenario]["metrics"]
    row = {"Scenario": SCENARIO_LABELS[scenario]}
    for key in delta_keys:
        short = UTILITY_NAMES[key]
        row[f"Scenario-Matched Original {short}"] = original[key]
        row[f"Full Retraining {short}"] = metrics[key]
        row[f"Δ {short}"] = metrics[key] - original[key]
    full_delta_rows.append(row)
full_delta_table = pd.DataFrame(full_delta_rows)
display(full_delta_table.style.format({
    column: "{:+.4f}" if column.startswith("Δ") else "{:.4f}"
    for column in full_delta_table.columns if column != "Scenario"
}))

change_summary = pd.DataFrame([
    {
        "Metric": UTILITY_NAMES[key],
        "Largest absolute change": full_delta_table[f"Δ {UTILITY_NAMES[key]}"].abs().max(),
        "Scenario": full_delta_table.loc[
            full_delta_table[f"Δ {UTILITY_NAMES[key]}"].abs().idxmax(), "Scenario"
        ],
    }
    for key in delta_keys
])
display(change_summary.round(4))

,Scenario,Scenario-Matched Original PR-AUC,Full Retraining PR-AUC,Δ PR-AUC,Scenario-Matched Original Balanced Accuracy,Full Retraining Balanced Accuracy,Δ Balanced Accuracy,Scenario-Matched Original BCE,Full Retraining BCE,Δ BCE,Scenario-Matched Original F1,Full Retraining F1,Δ F1,Scenario-Matched Original AUROC,Full Retraining AUROC,Δ AUROC
0,Recipient Withdrawal,0.1778,0.1833,+0.0055,0.6059,0.5681,-0.0378,0.4316,0.3532,-0.0785,0.2335,0.2048,-0.0287,0.7298,0.7332,+0.0034
1,Donor Withdrawal,0.1761,0.1753,-0.0008,0.6053,0.5439,-0.0614,0.4330,0.3190,-0.1140,0.2324,0.1581,-0.0742,0.7277,0.7286,+0.0010
2,Invalid Consent,0.1737,0.1725,-0.0013,0.6037,0.5644,-0.0393,0.4171,0.3494,-0.0677,0.2264,0.1919,-0.0346,0.7345,0.7360,+0.0015
3,Hospital Removal,0.1781,0.1833,+0.0052,0.6082,0.5611,-0.0472,0.4326,0.3542,-0.0783,0.2370,0.1932,-0.0438,0.7342,0.7395,+0.0053
4,Retention Expiry,0.1738,0.1615,-0.0123,0.5945,0.5493,-0.0453,0.3989,0.3450,-0.0539,0.2257,0.1677,-0.0579,0.7434,0.7352,-0.0082


,Metric,Largest absolute change,Scenario
0,PR-AUC,0.0123,Retention Expiry
1,Balanced Accuracy,0.0614,Donor Withdrawal
2,BCE,0.1140,Donor Withdrawal
3,F1,0.0742,Donor Withdrawal
4,AUROC,0.0082,Retention Expiry


## 2.4 Full Retraining Result Summary

The tables report how retained utility changes when each deletion request is removed and Qwen is rebuilt. The change summary identifies the largest absolute scenario-level change for each metric without introducing approximate-unlearning conclusions.

# 3. Shared Experiment Setup

Original Qwen answers “what was the model like before deletion?” Full Retraining answers “what would it look like if rebuilt without the forget data?” Approximate unlearning asks whether the first can move towards the second without full rebuilding.

## 3.1 Frozen Dataset and Splits

In [230]:
DATA_PATH = FINAL_SUBMISSION / "data" / "final" / "kidney_transplant_assessments.csv"
FEATURE_PATH = FINAL_SUBMISSION / "data" / "final" / "classifier_feature_list.json"
SPLIT_PATH = FINAL_SUBMISSION / "processed_data" / "split_assignments.csv"
MEMBERSHIP_PATH = FINAL_SUBMISSION / "processed_data" / "deletion_scenario_membership.csv"

for path in [DATA_PATH, FEATURE_PATH, SPLIT_PATH, MEMBERSHIP_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing frozen project input: {path}")

In [231]:
assessments = pd.read_csv(DATA_PATH)
feature_contract = json.loads(FEATURE_PATH.read_text())
split_assignments = pd.read_csv(SPLIT_PATH)
membership = pd.read_csv(MEMBERSHIP_PATH)

TARGET = feature_contract["target"]
FEATURES = feature_contract["classifier_features"]
data = assessments.merge(
    split_assignments[["recipient_id", "donor_id", "split"]],
    on=["recipient_id", "donor_id"], how="left", validate="many_to_one",
)
assert data["split"].notna().all() and len(data) == 60_000

In [232]:
# Serialisation is needed only when a reviewed method is enabled.
if GPU_EXECUTION_REQUESTED:
    labels = serialisation_spec["display_labels"]
    binary = {"previous_transplant", "infection_indicator", "previous_rejection"}

    def format_value(feature, value):
        if pd.isna(value): return "missing"
        if feature in binary: return "yes" if int(value) == 1 else "no"
        if isinstance(value, (float, np.floating)):
            return f"{float(value):.4f}".rstrip("0").rstrip(".")
        return str(value).strip()

    data["text"] = data.apply(
        lambda row: "\n".join(
            f"{labels[name]}: {format_value(name, row[name])}." for name in FEATURES
        ), axis=1,
    )
else:
    data["text"] = ""  # Avoid unnecessary local-review serialisation.
data["label"] = data[TARGET].astype("int64")

## 3.2 Deletion Scenarios

In [233]:
# Use frozen membership instead of rebuilding deletion requests.
splits = {name: data.loc[data["split"].eq(name)].copy() for name in ["train", "validation", "test"]}
scenario_sets = {}

for scenario in SCENARIOS:
    current = membership.loc[membership["scenario"].eq(scenario)]
    ids = {
        kind: set(current.loc[current["membership_type"].eq(kind), "assessment_id"])
        for kind in ["training_forget", "deleted_validation", "deleted_test"]
    }
    scenario_sets[scenario] = {
        "training_forget": splits["train"].loc[splits["train"]["assessment_id"].isin(ids["training_forget"])].copy(),
        "retained_train": splits["train"].loc[~splits["train"]["assessment_id"].isin(ids["training_forget"])].copy(),
        "retained_validation": splits["validation"].loc[~splits["validation"]["assessment_id"].isin(ids["deleted_validation"])].copy(),
        "retained_test": splits["test"].loc[~splits["test"]["assessment_id"].isin(ids["deleted_test"])].copy(),
    }
    assert len(scenario_sets[scenario]["training_forget"]) == EXPECTED_FORGET_ROWS[scenario]

In [234]:
scenario_sizes = pd.DataFrame([
    {
        "Scenario": SCENARIO_LABELS[scenario],
        "Forget train": len(parts["training_forget"]),
        "Retained train": len(parts["retained_train"]),
        "Retained validation": len(parts["retained_validation"]),
        "Retained test": len(parts["retained_test"]),
    }
    for scenario, parts in scenario_sets.items()
])
for scenario, parts in scenario_sets.items():
    expected_rows = int(full_retraining[scenario]["complete"]["retained_test_rows"])
    assert len(parts["retained_test"]) == expected_rows
    assert len(scenario_original[scenario]["predictions"]) == expected_rows
display(scenario_sizes)

,Scenario,Forget train,Retained train,Retained validation,Retained test
0,Recipient Withdrawal,426,41598,8904,8898
1,Donor Withdrawal,1992,40032,8472,8496
2,Invalid Consent,4148,37876,8046,8078
3,Hospital Removal,4314,37710,8166,8124
4,Retention Expiry,6262,35762,7624,7614


## 3.3 Common Evaluation Metrics

Every method uses the frozen threshold 0.55. Re-selecting a threshold after unlearning could change threshold-dependent results and would make the comparison unfair.

In [235]:
if GPU_EXECUTION_REQUESTED:
    import copy
    import torch
    import torch.nn.functional as F
    from torch import nn
    from torch.utils.data import DataLoader, Dataset
    from safetensors.torch import load_file
    from scipy.stats import ks_2samp
    from tqdm.auto import tqdm
    from unsloth import FastVisionModel
    if not torch.cuda.is_available():
        raise RuntimeError("Approximate unlearning requires a CUDA GPU.")
    DEVICE = torch.device("cuda")

In [236]:
if GPU_EXECUTION_REQUESTED:
    def compute_utility(predictions):
        """Use the same frozen-probability metric implementation for new predictions."""
        return compute_saved_utility(predictions)

## 3.4 Truth Ratio and KS Evaluation

The project Truth Ratio is (p incorrect + ε) / (p true + ε). Approximate and Full Retraining Truth Ratio distributions are compared for the same scenario with a two-sample Kolmogorov–Smirnov test.

A lower KS statistic means closer behaviour. A higher p-value means less evidence of different distributions, but it does not prove successful forgetting.

In [237]:
def truth_ratio(labels, probabilities, epsilon=1e-12):
    """Match the final MLP definition exactly."""
    probability_true = np.where(labels == 1, probabilities, 1 - probabilities)
    probability_incorrect = 1 - probability_true
    return (probability_incorrect + epsilon) / (probability_true + epsilon)

In [238]:
if GPU_EXECUTION_REQUESTED:
    def calculate_ks(approximate, reference):
        """Compare aligned forget rows with the scenario Full Retraining reference."""
        joined = approximate.merge(
            reference, on="assessment_id",
            suffixes=("_approximate", "_full"), validate="one_to_one",
        )
        assert (joined["label_approximate"] == joined["label_full"]).all()
        labels = joined["label_approximate"].to_numpy()
        approx_ratio = truth_ratio(labels, joined["probability_class_1_approximate"].to_numpy())
        full_ratio = truth_ratio(labels, joined["probability_class_1_full"].to_numpy())
        result = ks_2samp(approx_ratio, full_ratio, alternative="two-sided", method="auto")
        return float(result.statistic), float(result.pvalue)

## 3.5 Frozen Qwen Model Loader

Instantiating the pretrained architecture is reconstruction, not baseline retraining. The saved LoRA tensors and saved classification head are then restored. No optimiser exists in this function.

In [239]:
if GPU_EXECUTION_REQUESTED:
    class FP32ClassificationHead(nn.Linear):
        """Keep the two-class head in FP32, matching baseline training."""
        def forward(self, hidden_states):
            return F.linear(hidden_states.float(), self.weight, self.bias)

In [240]:
if GPU_EXECUTION_REQUESTED:
    def build_qwen_architecture():
        # Rebuild the Qwen architecture first, then restore the frozen LoRA adapter.
        base, processor = FastVisionModel.from_pretrained(
            MODEL_ID, load_in_4bit=False, load_in_16bit=True,
            max_seq_length=MAX_SEQ_LENGTH, use_gradient_checkpointing=False,
        )
        tokenizer = getattr(processor, "tokenizer", processor)
        tokenizer.padding_side = "right"
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
        old_head = base.get_output_embeddings()
        # This temporary two-class structure is never used with random weights;
        # load_frozen_original_qwen overwrites it with the exact saved binary head.
        base.set_output_embeddings(FP32ClassificationHead(
            old_head.in_features, 2, bias=False,
            device=old_head.weight.device, dtype=torch.float32,
        ))
        base.config.num_labels = 2
        base.config.pad_token_id = tokenizer.pad_token_id
        return base, processor, tokenizer

In [241]:
if GPU_EXECUTION_REQUESTED:
    def attach_baseline_lora(base):
        saved_lora = adapter_configuration
        model = FastVisionModel.get_peft_model(
            base, finetune_vision_layers=False, finetune_language_layers=True,
            finetune_attention_modules=True, finetune_mlp_modules=True,
            target_modules=saved_lora["target_modules"],
            modules_to_save=saved_lora["modules_to_save"],
            r=int(saved_lora["r"]), lora_alpha=int(saved_lora["lora_alpha"]),
            lora_dropout=float(saved_lora["lora_dropout"]), bias=saved_lora["bias"],
            use_gradient_checkpointing=False, random_state=42,
            use_rslora=bool(saved_lora.get("use_rslora", False)),
            loftq_config=saved_lora.get("loftq_config") or None,
        )
        model.config.use_cache = False
        for parameter in model.parameters():
            if parameter.requires_grad:
                parameter.data = parameter.data.float()
        return model

In [242]:
if GPU_EXECUTION_REQUESTED:
    def map_adapter_state(model, saved_state):
        current = model.state_dict()
        converted = {}
        for saved_name, tensor in saved_state.items():
            candidates = [
                saved_name,
                saved_name.replace(".lora_A.weight", ".lora_A.default.weight"),
                saved_name.replace(".lora_B.weight", ".lora_B.default.weight"),
            ]
            if saved_name.endswith("lm_head.weight"):
                candidates += [
                    saved_name.replace("lm_head.weight", "lm_head.modules_to_save.default.weight"),
                    saved_name.replace("lm_head.weight", "lm_head.original_module.weight"),
                ]
            matches = [name for name in candidates if name in current]
            if not matches:
                raise RuntimeError(f"Could not map frozen tensor: {saved_name}")
            preferred = [name for name in matches if "modules_to_save.default" in name]
            converted[preferred[0] if preferred else matches[0]] = tensor
        return converted

In [243]:
if GPU_EXECUTION_REQUESTED:
    def load_frozen_original_qwen():
        """Return a fresh trainable copy of the saved Original Qwen classifier."""
        # This is model loading only; no baseline training occurs here.
        torch.manual_seed(42)
        base, processor, tokenizer = build_qwen_architecture()
        # Rebuild the Qwen architecture first, then restore the frozen LoRA adapter.
        model = attach_baseline_lora(base)
        adapter_state = load_file(str(baseline_paths["Adapter weights"]))
        model.load_state_dict(map_adapter_state(model, adapter_state), strict=False)

        # Restore the separately saved binary head so this is the exact trained baseline.
        head_state = torch.load(
            baseline_paths["Classification head"], map_location="cpu", weights_only=True,
        )
        if not head_state:
            raise RuntimeError("The frozen classification head is empty.")
        model.load_state_dict(head_state, strict=False)
        return model, processor, tokenizer

In [244]:
if GPU_EXECUTION_REQUESTED:
    class AssessmentDataset(Dataset):
        def __init__(self, frame):
            self.frame = frame.reset_index(drop=True)
        def __len__(self):
            return len(self.frame)
        def __getitem__(self, index):
            row = self.frame.iloc[index]
            return {
                "text": row["text"], "label": int(row["label"]),
                "assessment_id": str(row["assessment_id"]),
            }

In [245]:
if GPU_EXECUTION_REQUESTED:
    def make_collate(tokenizer):
        def collate(rows):
            encoded = tokenizer(
                [row["text"] for row in rows], padding=True, truncation=True,
                max_length=MAX_SEQ_LENGTH, return_tensors="pt",
            )
            return {
                **encoded, "labels": torch.tensor([row["label"] for row in rows]),
                "assessment_id": [row["assessment_id"] for row in rows],
            }
        return collate

    def make_loader(frame, tokenizer, batch_size, shuffle=False, seed=42):
        return DataLoader(
            AssessmentDataset(frame), batch_size=batch_size, shuffle=shuffle,
            generator=torch.Generator().manual_seed(seed),
            collate_fn=make_collate(tokenizer), num_workers=0,
        )

In [246]:
if GPU_EXECUTION_REQUESTED:
    def final_token_logits(model, batch):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        sequence = model(input_ids=input_ids, attention_mask=attention_mask).logits
        final_index = attention_mask.sum(dim=1) - 1
        return sequence[torch.arange(len(input_ids), device=DEVICE), final_index]

    def positive_class_weights(frame):
        labels = frame["label"].to_numpy()
        return torch.tensor(
            [1.0, (len(labels) - labels.sum()) / labels.sum()],
            dtype=torch.float32, device=DEVICE,
        )

In [247]:
if GPU_EXECUTION_REQUESTED:
    @torch.no_grad()
    def predict_frame(model, tokenizer, frame, description):
        model.eval()
        identifiers, labels, probabilities = [], [], []
        loader = make_loader(frame, tokenizer, batch_size=16)
        for batch in tqdm(loader, desc=description, leave=False):
            logits = final_token_logits(model, batch).float()
            identifiers.extend(batch["assessment_id"])
            labels.extend(batch["labels"].numpy())
            probabilities.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
        return pd.DataFrame({
            "assessment_id": identifiers, "label": labels,
            "probability_class_1": probabilities,
        })

In [248]:
def approximate_model_paths(method, scenario):
    model_dir = APPROXIMATE_MODEL_ROOTS[method] / scenario
    return {
        "root": model_dir,
        "adapter": model_dir / "adapter",
        "adapter_config": model_dir / "adapter" / "adapter_config.json",
        "adapter_weights": model_dir / "adapter" / "adapter_model.safetensors",
        "head": model_dir / "binary_classification_head.pt",
    }

def missing_approximate_model_files(method, scenario):
    paths = approximate_model_paths(method, scenario)
    return [paths[key] for key in ["adapter_config", "adapter_weights", "head"] if not paths[key].is_file()]

if GPU_EXECUTION_REQUESTED:
    def save_result(output_dir, model, processor, history, metrics, retained, forget, completion):
        # Save selected model artefacts first; COMPLETE.json must certify both model and evidence.
        model_paths = approximate_model_paths(completion["method"], completion["scenario"])
        model_paths["root"].mkdir(parents=True, exist_ok=True)
        model.save_pretrained(model_paths["adapter"], safe_serialization=True)
        processor.save_pretrained(model_paths["adapter"])
        head_state = {
            name: tensor.detach().cpu() for name, tensor in model.state_dict().items()
            if "lm_head" in name
        }
        if not head_state:
            raise RuntimeError("The trained binary classification head was not found.")
        torch.save(head_state, model_paths["head"])
        missing_model = missing_approximate_model_files(completion["method"], completion["scenario"])
        if missing_model:
            raise FileNotFoundError("Final approximate model artefact is incomplete: " + "; ".join(map(str, missing_model)))

        # Save result evidence only after the final model bundle is safely present.
        history.to_csv(output_dir / "training_history.csv", index=False)
        pd.DataFrame([metrics]).to_csv(output_dir / "retained_test_metrics.csv", index=False)
        retained.to_csv(output_dir / "retained_test_probabilities.csv", index=False)
        forget.to_csv(output_dir / "forget_set_probabilities.csv", index=False)
        (output_dir / "configuration.json").write_text(json.dumps(completion["configuration"], indent=2))
        (output_dir / "runtime.json").write_text(json.dumps({
            "training_seconds": completion["training_seconds"],
            "peak_gpu_memory_gib": completion["peak_gpu_memory_gib"],
        }, indent=2))

        # Written last: this marker means both model artefacts and result evidence are complete.
        (output_dir / "COMPLETE.json").write_text(json.dumps(completion, indent=2))

## 3.6 Verify Loaded Original Qwen

Before any update, a temporary reconstruction is compared with a deterministic subset of saved baseline predictions. This is inference-only. A mismatch stops execution and the temporary model is deleted.

In [249]:
if GPU_EXECUTION_REQUESTED:
    def verify_loaded_original_qwen(sample_size=64):
        if baseline_probabilities.empty:
            raise FileNotFoundError("Saved baseline probabilities are required for reconstruction verification.")
        probe_ids = baseline_probabilities["assessment_id"].astype(str).head(sample_size)
        assert len(probe_ids) == sample_size and probe_ids.is_unique
        probe = splits["test"].set_index("assessment_id").loc[probe_ids].reset_index()
        expected = baseline_probabilities.set_index("assessment_id").loc[probe_ids]
        assert probe["label"].to_numpy().tolist() == expected["label"].to_numpy().tolist()
        model, processor, tokenizer = load_frozen_original_qwen()
        observed = predict_frame(model, tokenizer, probe, "Verify frozen Original Qwen")
        difference = np.max(np.abs(
            observed["probability_class_1"].to_numpy()
            - expected["probability_class_1"].to_numpy()
        ))
        passed = np.allclose(
            observed["probability_class_1"], expected["probability_class_1"],
            atol=1e-5, rtol=1e-4,
        )
        del model, processor, tokenizer
        gc.collect()
        torch.cuda.empty_cache()
        if not passed:
            raise RuntimeError(f"Frozen baseline reconstruction mismatch: {difference:.8f}")
        return float(difference)

In [250]:
BASELINE_RECONSTRUCTION_VERIFIED = False
if GPU_EXECUTION_REQUESTED:
    maximum_reconstruction_difference = verify_loaded_original_qwen()
    BASELINE_RECONSTRUCTION_VERIFIED = True
    print(f"Frozen Original Qwen verified; maximum difference {maximum_reconstruction_difference:.8f}")
else:
    print("LOCAL REVIEW MODE: baseline reconstruction verification is deferred until GPU execution.")

LOCAL REVIEW MODE: baseline reconstruction verification is deferred until GPU execution.


In [251]:
def frozen_file_state():
    paths = [path for path in baseline_paths.values() if path.is_file()]
    for scenario in SCENARIOS:
        paths.extend((FULL_RETRAINING_ROOT / scenario / name) for name in FULL_FILES)
        full_model_dir = FULL_RETRAINING_MODEL_ROOT / scenario
        paths.extend([
            full_model_dir / "adapter" / "adapter_config.json",
            full_model_dir / "adapter" / "adapter_model.safetensors",
            full_model_dir / "adapter" / "processor_config.json",
            full_model_dir / "adapter" / "tokenizer.json",
            full_model_dir / "adapter" / "tokenizer_config.json",
            full_model_dir / "binary_classification_head.pt",
        ])
    return {
        str(path): (path.stat().st_size, path.stat().st_mtime_ns)
        for path in paths if path.is_file()
    }

FROZEN_STATE_BEFORE = frozen_file_state()

# 4. Retain-Set Fine-Tuning

## 4.1 Method

This follows the final MLP implementation in 03_retain_set_fine_tuning.ipynb. It starts from trained Original Qwen, removes the scenario’s forget rows, and minimises weighted cross-entropy on retained training data only:

L_RSFT(θ) = L_retain(θ).

The Qwen implementation keeps its architecture-specific LoRA learning rate and effective batch size, while matching the final MLP stopping schedule: up to 30 epochs with early-stopping patience 5. Selection uses retained-validation weighted loss only.

## 4.2 Configuration

In [252]:
RSFT_CONFIG = {
    "learning_rate": 1e-6,
    "batch_size": 8,
    "gradient_accumulation": 4,
    "effective_batch_size": 32,
    "maximum_epochs": 30,
    "weight_decay": 1e-4,
    "gradient_clip_norm": 1.0,
    "lora_rank": 16,
    "lora_alpha": 16,
    "lora_dropout": 0,
    "selection": "minimum retained-validation weighted cross-entropy",
    "early_stopping_patience": 5,
    "mlp_reference": "03_retain_set_fine_tuning.ipynb",
}
display(pd.Series(RSFT_CONFIG, name="Value").to_frame())

,Value
learning_rate,0.000001
batch_size,8
gradient_accumulation,4
effective_batch_size,32
maximum_epochs,30
weight_decay,0.0001
gradient_clip_norm,1.0
lora_rank,16
lora_alpha,16
lora_dropout,0


Checkpoint selection matches the final MLP Retain-Set Fine-Tuning procedure: minimise retained-validation weighted cross-entropy, allow up to 30 epochs, and stop after 5 consecutive epochs without improvement. Neither retained-test results nor forget-set results influence selection.

## 4.3 Prepare Retained Data

In [253]:
# Remove forget rows completely before fine-tuning.
rsft_data_summary = scenario_sizes[
    ["Scenario", "Forget train", "Retained train", "Retained validation", "Retained test"]
].copy()
display(rsft_data_summary)

,Scenario,Forget train,Retained train,Retained validation,Retained test
0,Recipient Withdrawal,426,41598,8904,8898
1,Donor Withdrawal,1992,40032,8472,8496
2,Invalid Consent,4148,37876,8046,8078
3,Hospital Removal,4314,37710,8166,8124
4,Retention Expiry,6262,35762,7624,7614


## 4.4 Load Fresh Original Qwen

Every scenario calls load_frozen_original_qwen() independently, so updates from one deletion request cannot affect another experiment.

In [254]:
if RUN_RETAIN_SET_FINE_TUNING:
    def train_retain_set_fine_tuning(model, tokenizer, parts, scenario):
        train_loader = make_loader(parts["retained_train"], tokenizer, RSFT_CONFIG["batch_size"], True)
        weights = positive_class_weights(parts["retained_train"])
        trainable = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(
            trainable, lr=RSFT_CONFIG["learning_rate"], weight_decay=RSFT_CONFIG["weight_decay"],
        )
        best_loss, best_state, stale, history = np.inf, None, 0, []
        started = time.perf_counter()

        for epoch in range(1, RSFT_CONFIG["maximum_epochs"] + 1):
            model.train()
            total, rows = 0.0, 0
            bar = tqdm(train_loader, desc=f"Retain-Set FT | {scenario} | epoch {epoch}")
            for step, batch in enumerate(bar, start=1):
                labels = batch["labels"].to(DEVICE)
                loss = F.cross_entropy(final_token_logits(model, batch).float(), labels, weight=weights)
                (loss / RSFT_CONFIG["gradient_accumulation"]).backward()
                if step % RSFT_CONFIG["gradient_accumulation"] == 0 or step == len(train_loader):
                    torch.nn.utils.clip_grad_norm_(trainable, RSFT_CONFIG["gradient_clip_norm"])
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
                total += float(loss.detach()) * len(labels)
                rows += len(labels)
                bar.set_postfix(loss=f"{total / rows:.4f}")
            validation = predict_frame(model, tokenizer, parts["retained_validation"], "RSFT validation")
            val_probability = validation["probability_class_1"].to_numpy()
            val_labels = validation["label"].to_numpy()
            row_weight = np.where(val_labels == 1, float(weights[1].cpu()), 1.0)
            per_row_bce = -(val_labels * np.log(np.clip(val_probability, 1e-12, 1))
                            + (1 - val_labels) * np.log(np.clip(1 - val_probability, 1e-12, 1)))
            val_loss = float(np.average(per_row_bce, weights=row_weight))
            improved = val_loss < best_loss - 1e-12
            history.append({"epoch": epoch, "training_weighted_ce": total / rows,
                            "validation_bce": val_loss, "selected": improved})
            if improved:
                best_loss, best_state, stale = val_loss, copy.deepcopy(model.state_dict()), 0
            else:
                stale += 1
            if stale >= RSFT_CONFIG["early_stopping_patience"]:
                break
        model.load_state_dict(best_state)
        return model, pd.DataFrame(history), time.perf_counter() - started

## 4.5 Run Five Scenarios

In [255]:
if RUN_UNLEARNING and RUN_RETAIN_SET_FINE_TUNING:
    assert RUN_UNLEARNING and BASELINE_RECONSTRUCTION_VERIFIED
    for index, scenario in enumerate(SCENARIOS, start=1):
        output = UNLEARNING_ROOT / "retain_set_fine_tuning" / scenario
        model_output = APPROXIMATE_MODEL_ROOTS["retain_set_fine_tuning"] / scenario
        complete = output / "COMPLETE.json"
        print(f"Retain-Set Fine-Tuning | Scenario {index}/5: {SCENARIO_LABELS[scenario]}")
        if complete.is_file() and json.loads(complete.read_text()).get("status") == "complete":
            missing_model = missing_approximate_model_files("retain_set_fine_tuning", scenario)
            if missing_model:
                raise FileNotFoundError("Result evidence is complete but the final model artefact is incomplete: " + "; ".join(map(str, missing_model)))
            print("Completed result and model bundle loaded; training skipped.")
            continue
        if output.exists() or model_output.exists():
            raise FileExistsError(f"Inspect partial result/model without overwriting: {output}; {model_output}")
        output.mkdir(parents=True, exist_ok=False)
        model_output.mkdir(parents=True, exist_ok=False)
        torch.cuda.reset_peak_memory_stats()
        model, processor, tokenizer = load_frozen_original_qwen()
        model, history, seconds = train_retain_set_fine_tuning(
            model, tokenizer, scenario_sets[scenario], SCENARIO_LABELS[scenario],
        )
        retained = predict_frame(model, tokenizer, scenario_sets[scenario]["retained_test"], "RSFT retained test")
        forget = predict_frame(model, tokenizer, scenario_sets[scenario]["training_forget"], "RSFT forget set")
        ks_stat, ks_p = calculate_ks(forget, full_retraining[scenario]["forget"])
        completion = {"status": "complete", "method": "retain_set_fine_tuning",
            "scenario": scenario, "training_seconds": seconds,
            "peak_gpu_memory_gib": torch.cuda.max_memory_allocated() / 1024**3,
            "ks_statistic": ks_stat, "ks_p_value": ks_p, "configuration": RSFT_CONFIG}
        save_result(output, model, processor, history, compute_utility(retained), retained, forget, completion)
        del model, processor, tokenizer
        gc.collect()
        torch.cuda.empty_cache()
else:
    print("Retain-Set Fine-Tuning training is disabled.")

Retain-Set Fine-Tuning training is disabled.


In [256]:
def load_method_results(method):
    results = {}
    for scenario in SCENARIOS:
        folder = UNLEARNING_ROOT / method / scenario
        complete_path = folder / "COMPLETE.json"
        metrics_path = folder / "retained_test_metrics.csv"
        if not complete_path.is_file():
            continue
        complete = json.loads(complete_path.read_text())
        if complete.get("status") != "complete" or not metrics_path.is_file():
            raise ValueError(f"Incomplete saved result: {folder}")
        missing_model = missing_approximate_model_files(method, scenario)
        if missing_model:
            raise FileNotFoundError(
                "Result evidence exists but the final model artefact is incomplete: "
                + "; ".join(map(str, missing_model))
            )
        results[scenario] = {
            "complete": complete,
            "metrics": pd.read_csv(metrics_path).iloc[0],
        }
    return results

In [257]:
def method_comparison_tables(method, label):
    saved = load_method_results(method)
    utility_rows, forgetting_rows, delta_rows = [], [], []
    for scenario in SCENARIOS:
        original = scenario_original[scenario]["metrics"]
        reference = full_retraining[scenario]["metrics"]
        for model_label, metrics in [("Scenario-Matched Original Qwen", original), ("Full Retraining", reference)]:
            utility_rows.append({"Scenario": SCENARIO_LABELS[scenario], "Model": model_label,
                                 **{UTILITY_NAMES[k]: metrics[k] for k in UTILITY_KEYS}})
        if scenario in saved:
            approximate = saved[scenario]["metrics"]
            utility_rows.append({"Scenario": SCENARIO_LABELS[scenario], "Model": label,
                                 **{UTILITY_NAMES[k]: approximate[k] for k in UTILITY_KEYS}})
            forgetting_rows.append({"Scenario": SCENARIO_LABELS[scenario], "Method": label,
                "KS Statistic": saved[scenario]["complete"]["ks_statistic"],
                "KS p-value": saved[scenario]["complete"]["ks_p_value"]})
            delta = {"Scenario": SCENARIO_LABELS[scenario], "Method": label}
            for key in ["pr_auc", "balanced_accuracy", "binary_cross_entropy", "f1", "auroc"]:
                name = UTILITY_NAMES[key]
                delta[f"Δ {name} vs Original"] = approximate[key] - original[key]
                delta[f"Δ {name} vs Full Retraining"] = approximate[key] - reference[key]
            delta_rows.append(delta)
    return pd.DataFrame(utility_rows), pd.DataFrame(forgetting_rows), pd.DataFrame(delta_rows)

## 4.6–4.8 Utility, Forgetting, and Comparison

In [258]:
rsft_utility, rsft_forgetting, rsft_deltas = method_comparison_tables(
    "retain_set_fine_tuning", "Retain-Set Fine-Tuning",
)
display(rsft_utility.round(4))
display(rsft_forgetting.round(4) if not rsft_forgetting.empty else "No completed RSFT results.")
display(rsft_deltas.round(4) if not rsft_deltas.empty else "No completed RSFT deltas.")

,Scenario,Model,PR-AUC,Balanced Accuracy,BCE,F1,AUROC,Precision,Recall,Specificity
0,Recipient Withdrawal,Scenario-Matched Original Qwen,0.1778,0.6059,0.4316,0.2335,0.7298,0.1800,0.3323,0.8795
1,Recipient Withdrawal,Full Retraining,0.1833,0.5681,0.3532,0.2048,0.7332,0.2257,0.1875,0.9488
2,Donor Withdrawal,Scenario-Matched Original Qwen,0.1761,0.6053,0.4330,0.2324,0.7277,0.1785,0.3328,0.8777
3,Donor Withdrawal,Full Retraining,0.1753,0.5439,0.3190,0.1581,0.7286,0.2630,0.1131,0.9747
4,Invalid Consent,Scenario-Matched Original Qwen,0.1737,0.6037,0.4171,0.2264,0.7345,0.1739,0.3246,0.8829
5,Invalid Consent,Full Retraining,0.1725,0.5644,0.3494,0.1919,0.7360,0.1981,0.1860,0.9429
6,Hospital Removal,Scenario-Matched Original Qwen,0.1781,0.6082,0.4326,0.2370,0.7342,0.1822,0.3388,0.8776
7,Hospital Removal,Full Retraining,0.1833,0.5611,0.3542,0.1932,0.7395,0.2262,0.1686,0.9536
8,Retention Expiry,Scenario-Matched Original Qwen,0.1738,0.5945,0.3989,0.2257,0.7434,0.1852,0.2888,0.9003
9,Retention Expiry,Full Retraining,0.1615,0.5493,0.3450,0.1677,0.7352,0.2074,0.1408,0.9578


'No completed RSFT results.'

'No completed RSFT deltas.'

## 4.9 Findings

Findings will be added after all five Retain-Set Fine-Tuning scenario results are available. BCE deltas must be interpreted in the opposite direction from higher-is-better metrics: a negative BCE delta may be favourable.

# 5. Gradient Ascent

## 5.1 Method

This follows the final MLP implementation in 04_gradient_ascent.ipynb:

L_GA(θ) = −L_forget(θ).

θ denotes the trainable LoRA/head parameters and L_forget is forget-set cross-entropy. Minimising its negative increases ordinary forget loss. The Qwen proposal reduces learning rate and maximum epochs for LoRA stability, while retaining the MLP 5% retained-validation safety boundary and three-breach stopping rule.

## 5.2 Configuration

In [259]:
GA_CONFIG = {
    "learning_rate": 1e-6,
    "batch_size": 8,
    "gradient_accumulation": 4,
    "effective_batch_size": 32,
    "maximum_epochs": 5,
    "weight_decay": 0.0,
    "gradient_clip_norm": 1.0,
    "lora_rank": 16,
    "lora_alpha": 16,
    "lora_dropout": 0.0,
    "validation_relative_allowance": 0.05,
    "safety_patience": 3,
    "selection": "largest forget BCE within retained-validation 5% loss boundary",
    "mlp_reference": "04_gradient_ascent.ipynb",
}
display(pd.Series(GA_CONFIG, name="Value").to_frame())

,Value
learning_rate,0.000001
batch_size,8
gradient_accumulation,4
effective_batch_size,32
maximum_epochs,5
weight_decay,0.0
gradient_clip_norm,1.0
lora_rank,16
lora_alpha,16
lora_dropout,0.0


Checkpoint selection: maximise complete forget-set BCE among checkpoints whose retained-validation weighted BCE is no more than 5% above the reconstructed baseline. Stop after three consecutive boundary breaches. If no updated checkpoint is eligible, retain epoch 0. Test and final KS results never select a checkpoint.

## 5.3 Prepare Forget Data

In [260]:
# Gradient Ascent updates use only the authoritative training-forget rows.
ga_data_summary = scenario_sizes[["Scenario", "Forget train", "Retained validation"]].copy()
display(ga_data_summary)

,Scenario,Forget train,Retained validation
0,Recipient Withdrawal,426,8904
1,Donor Withdrawal,1992,8472
2,Invalid Consent,4148,8046
3,Hospital Removal,4314,8166
4,Retention Expiry,6262,7624


## 5.4 Training Implementation

In [261]:
if RUN_GRADIENT_ASCENT:
    def frame_bce(model, tokenizer, frame, weights=None):
        predicted = predict_frame(model, tokenizer, frame, "GA loss check")
        probability = np.clip(predicted["probability_class_1"].to_numpy(), 1e-12, 1 - 1e-12)
        labels = predicted["label"].to_numpy()
        if weights is None:
            return log_loss(labels, probability, labels=[0, 1])
        row_weight = np.where(labels == 1, float(weights[1].cpu()), float(weights[0].cpu()))
        loss = -(labels * np.log(probability) + (1 - labels) * np.log(1 - probability))
        return float(np.average(loss, weights=row_weight))

In [262]:
if RUN_GRADIENT_ASCENT:
    def train_gradient_ascent(model, tokenizer, parts, scenario):
        loader = make_loader(parts["training_forget"], tokenizer, GA_CONFIG["batch_size"], True)
        weights = positive_class_weights(parts["retained_train"])
        trainable = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(trainable, lr=GA_CONFIG["learning_rate"], weight_decay=0)
        baseline_validation = frame_bce(model, tokenizer, parts["retained_validation"], weights)
        validation_limit = baseline_validation * (1 + GA_CONFIG["validation_relative_allowance"])
        best_state, best_forget, best_epoch = copy.deepcopy(model.state_dict()), -np.inf, 0
        breaches, history, started = 0, [], time.perf_counter()

        for epoch in range(1, GA_CONFIG["maximum_epochs"] + 1):
            model.train()
            bar = tqdm(loader, desc=f"Gradient Ascent | {scenario} | epoch {epoch}")
            for step, batch in enumerate(bar, start=1):
                labels = batch["labels"].to(DEVICE)
                forget_loss = F.cross_entropy(final_token_logits(model, batch).float(), labels)
                objective = -forget_loss  # Reverse ordinary forget-set optimisation.
                (objective / GA_CONFIG["gradient_accumulation"]).backward()
                if step % GA_CONFIG["gradient_accumulation"] == 0 or step == len(loader):
                    torch.nn.utils.clip_grad_norm_(trainable, GA_CONFIG["gradient_clip_norm"])
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
                bar.set_postfix(forget=f"{forget_loss.item():.4f}", objective=f"{objective.item():.4f}")
            complete_forget = frame_bce(model, tokenizer, parts["training_forget"])
            validation = frame_bce(model, tokenizer, parts["retained_validation"], weights)
            eligible = np.isfinite([complete_forget, validation]).all() and validation <= validation_limit
            history.append({"epoch": epoch, "forget_bce": complete_forget,
                            "validation_weighted_bce": validation, "eligible": eligible})
            if eligible and complete_forget > best_forget:
                best_state, best_forget, best_epoch = copy.deepcopy(model.state_dict()), complete_forget, epoch
            breaches = 0 if eligible else breaches + 1
            if breaches >= GA_CONFIG["safety_patience"]:
                break
        model.load_state_dict(best_state)
        return model, pd.DataFrame(history), time.perf_counter() - started, best_epoch

## 5.5 Run Five Scenarios

In [263]:
if RUN_UNLEARNING and RUN_GRADIENT_ASCENT:
    assert RUN_UNLEARNING and BASELINE_RECONSTRUCTION_VERIFIED
    for index, scenario in enumerate(SCENARIOS, start=1):
        output = UNLEARNING_ROOT / "gradient_ascent" / scenario
        model_output = APPROXIMATE_MODEL_ROOTS["gradient_ascent"] / scenario
        complete = output / "COMPLETE.json"
        print(f"Gradient Ascent | Scenario {index}/5: {SCENARIO_LABELS[scenario]}")
        if complete.is_file() and json.loads(complete.read_text()).get("status") == "complete":
            missing_model = missing_approximate_model_files("gradient_ascent", scenario)
            if missing_model:
                raise FileNotFoundError("Result evidence is complete but the final model artefact is incomplete: " + "; ".join(map(str, missing_model)))
            print("Completed result and model bundle loaded; training skipped.")
            continue
        if output.exists() or model_output.exists():
            raise FileExistsError(f"Inspect partial result/model without overwriting: {output}; {model_output}")
        output.mkdir(parents=True, exist_ok=False)
        model_output.mkdir(parents=True, exist_ok=False)
        torch.cuda.reset_peak_memory_stats()
        model, processor, tokenizer = load_frozen_original_qwen()
        model, history, seconds, selected_epoch = train_gradient_ascent(
            model, tokenizer, scenario_sets[scenario], SCENARIO_LABELS[scenario],
        )
        retained = predict_frame(model, tokenizer, scenario_sets[scenario]["retained_test"], "GA retained test")
        forget = predict_frame(model, tokenizer, scenario_sets[scenario]["training_forget"], "GA forget set")
        ks_stat, ks_p = calculate_ks(forget, full_retraining[scenario]["forget"])
        completion = {"status": "complete", "method": "gradient_ascent",
            "scenario": scenario, "selected_epoch": selected_epoch,
            "training_seconds": seconds,
            "peak_gpu_memory_gib": torch.cuda.max_memory_allocated() / 1024**3,
            "ks_statistic": ks_stat, "ks_p_value": ks_p, "configuration": GA_CONFIG}
        save_result(output, model, processor, history, compute_utility(retained), retained, forget, completion)
        del model, processor, tokenizer
        gc.collect()
        torch.cuda.empty_cache()
else:
    print("Gradient Ascent training is disabled.")

Gradient Ascent training is disabled.


## 5.6–5.8 Utility, Forgetting, and Comparison

In [264]:
ga_utility, ga_forgetting, ga_deltas = method_comparison_tables(
    "gradient_ascent", "Gradient Ascent",
)
display(ga_utility.round(4))
display(ga_forgetting.round(4) if not ga_forgetting.empty else "No completed Gradient Ascent results.")
display(ga_deltas.round(4) if not ga_deltas.empty else "No completed Gradient Ascent deltas.")

,Scenario,Model,PR-AUC,Balanced Accuracy,BCE,F1,AUROC,Precision,Recall,Specificity
0,Recipient Withdrawal,Scenario-Matched Original Qwen,0.1778,0.6059,0.4316,0.2335,0.7298,0.1800,0.3323,0.8795
1,Recipient Withdrawal,Full Retraining,0.1833,0.5681,0.3532,0.2048,0.7332,0.2257,0.1875,0.9488
2,Donor Withdrawal,Scenario-Matched Original Qwen,0.1761,0.6053,0.4330,0.2324,0.7277,0.1785,0.3328,0.8777
3,Donor Withdrawal,Full Retraining,0.1753,0.5439,0.3190,0.1581,0.7286,0.2630,0.1131,0.9747
4,Invalid Consent,Scenario-Matched Original Qwen,0.1737,0.6037,0.4171,0.2264,0.7345,0.1739,0.3246,0.8829
5,Invalid Consent,Full Retraining,0.1725,0.5644,0.3494,0.1919,0.7360,0.1981,0.1860,0.9429
6,Hospital Removal,Scenario-Matched Original Qwen,0.1781,0.6082,0.4326,0.2370,0.7342,0.1822,0.3388,0.8776
7,Hospital Removal,Full Retraining,0.1833,0.5611,0.3542,0.1932,0.7395,0.2262,0.1686,0.9536
8,Retention Expiry,Scenario-Matched Original Qwen,0.1738,0.5945,0.3989,0.2257,0.7434,0.1852,0.2888,0.9003
9,Retention Expiry,Full Retraining,0.1615,0.5493,0.3450,0.1677,0.7352,0.2074,0.1408,0.9578


'No completed Gradient Ascent results.'

'No completed Gradient Ascent deltas.'

## 5.9 Findings

Findings will be added after all five Gradient Ascent scenario results are available.

# 6. Gradient Difference

## 6.1 Method

The final MLP implementation in 06_gradient_difference.ipynb uses:

L_GD(θ) = L_retain(θ) − λL_forget(θ),

where θ is the trainable state, L_retain preserves retained examples, L_forget is increased to weaken forgotten examples, and λ controls forget pressure.

The final MLP uses λ=1 for five fixed epochs. The proposed Qwen configuration instead carries forward the project’s prior stable Qwen LoRA setting (learning rate 4e-6, λ=0.1, two fixed epochs). This is a methodological difference, motivated by transformer/LoRA stability, but it must be explicitly approved before GPU execution.

## 6.2 Configuration

In [265]:
GD_CONFIG = {
    "learning_rate": 4e-6,
    "batch_size": 8,
    "gradient_accumulation": 4,
    "effective_batch_size": 32,
    "epochs": 2,
    "weight_decay": 0.0,
    "gradient_clip_norm": 1.0,
    "lora_rank": 16,
    "lora_alpha": 16,
    "lora_dropout": 0.0,
    "forget_weight_lambda": 0.1,
    "selection": "fixed final epoch; no checkpoint selection",
    "qwen_precedent": "previously approved fixed Qwen GD configuration",
    "mlp_reference": "06_gradient_difference.ipynb",
}
display(pd.Series(GD_CONFIG, name="Value").to_frame())

,Value
learning_rate,0.000004
batch_size,8
gradient_accumulation,4
effective_batch_size,32
epochs,2
weight_decay,0.0
gradient_clip_norm,1.0
lora_rank,16
lora_alpha,16
lora_dropout,0.0


Model selection: none. Epoch 2 is fixed before final evaluation. This preserves the prior Qwen stability configuration rather than choosing an epoch using retained-test utility, forget-set KS, or proximity to Full Retraining. The MLP likewise uses a fixed final epoch, but uses five epochs and λ=1.

## 6.3 Prepare Retain and Forget Data

In [266]:
# Each forget batch is paired with a retained batch.
# Cycle retained batches because set sizes differ.
gd_data_summary = scenario_sizes[["Scenario", "Forget train", "Retained train"]].copy()
gd_data_summary["Pairing"] = "cycle retained loader"
display(gd_data_summary)

,Scenario,Forget train,Retained train,Pairing
0,Recipient Withdrawal,426,41598,cycle retained loader
1,Donor Withdrawal,1992,40032,cycle retained loader
2,Invalid Consent,4148,37876,cycle retained loader
3,Hospital Removal,4314,37710,cycle retained loader
4,Retention Expiry,6262,35762,cycle retained loader


## 6.4 Training Implementation

In [267]:
if RUN_GRADIENT_DIFFERENCE:
    def next_retain_batch(iterator, loader):
        try:
            return next(iterator), iterator
        except StopIteration:
            iterator = iter(loader)
            return next(iterator), iterator

In [268]:
if RUN_GRADIENT_DIFFERENCE:
    def train_gradient_difference(model, tokenizer, parts, scenario):
        forget_loader = make_loader(parts["training_forget"], tokenizer, GD_CONFIG["batch_size"], True)
        retain_loader = make_loader(parts["retained_train"], tokenizer, GD_CONFIG["batch_size"], True)
        retain_iterator = iter(retain_loader)
        weights = positive_class_weights(parts["retained_train"])
        trainable = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(trainable, lr=GD_CONFIG["learning_rate"], weight_decay=0)
        history, started = [], time.perf_counter()

        for epoch in range(1, GD_CONFIG["epochs"] + 1):
            model.train()
            bar = tqdm(forget_loader, desc=f"Gradient Difference | {scenario} | epoch {epoch}")
            for step, forget_batch in enumerate(bar, start=1):
                retain_batch, retain_iterator = next_retain_batch(retain_iterator, retain_loader)
                forget_labels = forget_batch["labels"].to(DEVICE)
                retain_labels = retain_batch["labels"].to(DEVICE)
                forget_loss = F.cross_entropy(
                    final_token_logits(model, forget_batch).float(), forget_labels, weight=weights,
                )
                retain_loss = F.cross_entropy(
                    final_token_logits(model, retain_batch).float(), retain_labels, weight=weights,
                )
                objective = retain_loss - GD_CONFIG["forget_weight_lambda"] * forget_loss
                (objective / GD_CONFIG["gradient_accumulation"]).backward()
                if step % GD_CONFIG["gradient_accumulation"] == 0 or step == len(forget_loader):
                    torch.nn.utils.clip_grad_norm_(trainable, GD_CONFIG["gradient_clip_norm"])
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
                history.append({"epoch": epoch, "step": step, "retain_loss": float(retain_loss.detach()),
                                "forget_loss": float(forget_loss.detach()), "objective": float(objective.detach())})
                bar.set_postfix(retain=f"{retain_loss.item():.4f}",
                                forget=f"{forget_loss.item():.4f}", objective=f"{objective.item():.4f}")
        return model, pd.DataFrame(history), time.perf_counter() - started

## 6.5 Run Five Scenarios

In [269]:
if RUN_UNLEARNING and RUN_GRADIENT_DIFFERENCE:
    assert RUN_UNLEARNING and BASELINE_RECONSTRUCTION_VERIFIED
    for index, scenario in enumerate(SCENARIOS, start=1):
        output = UNLEARNING_ROOT / "gradient_difference" / scenario
        model_output = APPROXIMATE_MODEL_ROOTS["gradient_difference"] / scenario
        complete = output / "COMPLETE.json"
        print(f"Gradient Difference | Scenario {index}/5: {SCENARIO_LABELS[scenario]}")
        if complete.is_file() and json.loads(complete.read_text()).get("status") == "complete":
            missing_model = missing_approximate_model_files("gradient_difference", scenario)
            if missing_model:
                raise FileNotFoundError("Result evidence is complete but the final model artefact is incomplete: " + "; ".join(map(str, missing_model)))
            print("Completed result and model bundle loaded; training skipped.")
            continue
        if output.exists() or model_output.exists():
            raise FileExistsError(f"Inspect partial result/model without overwriting: {output}; {model_output}")
        output.mkdir(parents=True, exist_ok=False)
        model_output.mkdir(parents=True, exist_ok=False)
        torch.cuda.reset_peak_memory_stats()
        model, processor, tokenizer = load_frozen_original_qwen()
        model, history, seconds = train_gradient_difference(
            model, tokenizer, scenario_sets[scenario], SCENARIO_LABELS[scenario],
        )
        retained = predict_frame(model, tokenizer, scenario_sets[scenario]["retained_test"], "GD retained test")
        forget = predict_frame(model, tokenizer, scenario_sets[scenario]["training_forget"], "GD forget set")
        ks_stat, ks_p = calculate_ks(forget, full_retraining[scenario]["forget"])
        completion = {"status": "complete", "method": "gradient_difference",
            "scenario": scenario, "selected_epoch": GD_CONFIG["epochs"],
            "training_seconds": seconds,
            "peak_gpu_memory_gib": torch.cuda.max_memory_allocated() / 1024**3,
            "ks_statistic": ks_stat, "ks_p_value": ks_p, "configuration": GD_CONFIG}
        save_result(output, model, processor, history, compute_utility(retained), retained, forget, completion)
        del model, processor, tokenizer
        gc.collect()
        torch.cuda.empty_cache()
else:
    print("Gradient Difference training is disabled.")

Gradient Difference training is disabled.


## 6.6–6.8 Utility, Forgetting, and Comparison

In [270]:
gd_utility, gd_forgetting, gd_deltas = method_comparison_tables(
    "gradient_difference", "Gradient Difference",
)
display(gd_utility.round(4))
display(gd_forgetting.round(4) if not gd_forgetting.empty else "No completed Gradient Difference results.")
display(gd_deltas.round(4) if not gd_deltas.empty else "No completed Gradient Difference deltas.")

,Scenario,Model,PR-AUC,Balanced Accuracy,BCE,F1,AUROC,Precision,Recall,Specificity
0,Recipient Withdrawal,Scenario-Matched Original Qwen,0.1778,0.6059,0.4316,0.2335,0.7298,0.1800,0.3323,0.8795
1,Recipient Withdrawal,Full Retraining,0.1833,0.5681,0.3532,0.2048,0.7332,0.2257,0.1875,0.9488
2,Donor Withdrawal,Scenario-Matched Original Qwen,0.1761,0.6053,0.4330,0.2324,0.7277,0.1785,0.3328,0.8777
3,Donor Withdrawal,Full Retraining,0.1753,0.5439,0.3190,0.1581,0.7286,0.2630,0.1131,0.9747
4,Invalid Consent,Scenario-Matched Original Qwen,0.1737,0.6037,0.4171,0.2264,0.7345,0.1739,0.3246,0.8829
5,Invalid Consent,Full Retraining,0.1725,0.5644,0.3494,0.1919,0.7360,0.1981,0.1860,0.9429
6,Hospital Removal,Scenario-Matched Original Qwen,0.1781,0.6082,0.4326,0.2370,0.7342,0.1822,0.3388,0.8776
7,Hospital Removal,Full Retraining,0.1833,0.5611,0.3542,0.1932,0.7395,0.2262,0.1686,0.9536
8,Retention Expiry,Scenario-Matched Original Qwen,0.1738,0.5945,0.3989,0.2257,0.7434,0.1852,0.2888,0.9003
9,Retention Expiry,Full Retraining,0.1615,0.5493,0.3450,0.1677,0.7352,0.2074,0.1408,0.9578


'No completed Gradient Difference results.'

'No completed Gradient Difference deltas.'

## 6.9 Findings

Findings will be added after all five Gradient Difference scenario results are available.

# 7. Overall Approximate Unlearning Comparison

This section is intentionally last. It combines methods only after each has been introduced, implemented, and given its own result review.

## 7.1 All Methods

In [271]:
METHOD_LABELS = {
    "retain_set_fine_tuning": "Retain-Set Fine-Tuning",
    "gradient_ascent": "Gradient Ascent",
    "gradient_difference": "Gradient Difference",
}
all_rows = [{
    "Scenario": "Common baseline", "Method": "Original Qwen", "Forget Rows": "—",
    **{f"{UTILITY_NAMES[k]} {'↓' if k == 'binary_cross_entropy' else '↑'}": baseline_metrics.iloc[0][k]
       for k in UTILITY_KEYS},
    "KS Statistic ↓": "—", "KS p-value ↑": "—", "Training Time (minutes)": "—",
}]
for scenario in SCENARIOS:
    reference = full_retraining[scenario]
    matched_original = scenario_original[scenario]["metrics"]
    all_rows.append({
        "Scenario": SCENARIO_LABELS[scenario], "Method": "Scenario-Matched Original Qwen",
        "Forget Rows": f"{EXPECTED_FORGET_ROWS[scenario]:,}",
        **{f"{UTILITY_NAMES[k]} {'↓' if k == 'binary_cross_entropy' else '↑'}": matched_original[k]
           for k in UTILITY_KEYS},
        "KS Statistic ↓": "—", "KS p-value ↑": "—", "Training Time (minutes)": "—",
    })
    all_rows.append({
        "Scenario": SCENARIO_LABELS[scenario], "Method": "Full Retraining",
        "Forget Rows": f"{EXPECTED_FORGET_ROWS[scenario]:,}",
        **{f"{UTILITY_NAMES[k]} {'↓' if k == 'binary_cross_entropy' else '↑'}": reference["metrics"][k]
           for k in UTILITY_KEYS},
        "KS Statistic ↓": "Reference", "KS p-value ↑": "Reference",
        "Training Time (minutes)": reference["complete"]["training_seconds"] / 60,
    })
    for method, label in METHOD_LABELS.items():
        saved = load_method_results(method)
        if scenario not in saved:
            continue
        all_rows.append({
            "Scenario": SCENARIO_LABELS[scenario], "Method": label,
            "Forget Rows": f"{EXPECTED_FORGET_ROWS[scenario]:,}",
            **{f"{UTILITY_NAMES[k]} {'↓' if k == 'binary_cross_entropy' else '↑'}": saved[scenario]["metrics"][k]
               for k in UTILITY_KEYS},
            "KS Statistic ↓": saved[scenario]["complete"]["ks_statistic"],
            "KS p-value ↑": saved[scenario]["complete"]["ks_p_value"],
            "Training Time (minutes)": saved[scenario]["complete"]["training_seconds"] / 60,
        })
all_results = pd.DataFrame(all_rows)
display(all_results)

,Scenario,Method,Forget Rows,PR-AUC ↑,Balanced Accuracy ↑,BCE ↓,F1 ↑,AUROC ↑,Precision ↑,Recall ↑,Specificity ↑,KS Statistic ↓,KS p-value ↑,Training Time (minutes)
0,Common baseline,Original Qwen,—,0.176200,0.604858,0.432329,0.232435,0.728721,0.179153,0.330827,0.878890,—,—,—
1,Recipient Withdrawal,Scenario-Matched Original Qwen,426,0.177804,0.605918,0.431619,0.233530,0.729792,0.180017,0.332317,0.879520,—,—,—
2,Recipient Withdrawal,Full Retraining,426,0.183313,0.568149,0.353159,0.204829,0.733208,0.225688,0.187500,0.948799,Reference,Reference,174.945291
3,Donor Withdrawal,Scenario-Matched Original Qwen,"1,992",0.176064,0.605268,0.432956,0.232351,0.727668,0.178480,0.332803,0.877733,—,—,—
4,Donor Withdrawal,Full Retraining,"1,992",0.175302,0.543883,0.319004,0.158129,0.728626,0.262963,0.113057,0.974708,Reference,Reference,463.881006
5,Invalid Consent,Scenario-Matched Original Qwen,"4,148",0.173749,0.603743,0.417120,0.226438,0.734541,0.173872,0.324561,0.882925,—,—,—
6,Invalid Consent,Full Retraining,"4,148",0.172469,0.564413,0.349422,0.191855,0.736042,0.198131,0.185965,0.942861,Reference,Reference,517.403653
7,Hospital Removal,Scenario-Matched Original Qwen,"4,314",0.178066,0.608243,0.432551,0.236994,0.734223,0.182222,0.338843,0.877643,—,—,—
8,Hospital Removal,Full Retraining,"4,314",0.183297,0.561090,0.354245,0.193182,0.739540,0.226164,0.168595,0.953584,Reference,Reference,487.640731
9,Retention Expiry,Scenario-Matched Original Qwen,"6,262",0.173793,0.594546,0.398914,0.225670,0.743385,0.185185,0.288809,0.900283,—,—,—


## 7.2 Utility Comparison

In [272]:
utility_comparison = all_results.drop(columns=[
    "KS Statistic ↓", "KS p-value ↑",
])
display(utility_comparison)

,Scenario,Method,Forget Rows,PR-AUC ↑,Balanced Accuracy ↑,BCE ↓,F1 ↑,AUROC ↑,Precision ↑,Recall ↑,Specificity ↑,Training Time (minutes)
0,Common baseline,Original Qwen,—,0.176200,0.604858,0.432329,0.232435,0.728721,0.179153,0.330827,0.878890,—
1,Recipient Withdrawal,Scenario-Matched Original Qwen,426,0.177804,0.605918,0.431619,0.233530,0.729792,0.180017,0.332317,0.879520,—
2,Recipient Withdrawal,Full Retraining,426,0.183313,0.568149,0.353159,0.204829,0.733208,0.225688,0.187500,0.948799,174.945291
3,Donor Withdrawal,Scenario-Matched Original Qwen,"1,992",0.176064,0.605268,0.432956,0.232351,0.727668,0.178480,0.332803,0.877733,—
4,Donor Withdrawal,Full Retraining,"1,992",0.175302,0.543883,0.319004,0.158129,0.728626,0.262963,0.113057,0.974708,463.881006
5,Invalid Consent,Scenario-Matched Original Qwen,"4,148",0.173749,0.603743,0.417120,0.226438,0.734541,0.173872,0.324561,0.882925,—
6,Invalid Consent,Full Retraining,"4,148",0.172469,0.564413,0.349422,0.191855,0.736042,0.198131,0.185965,0.942861,517.403653
7,Hospital Removal,Scenario-Matched Original Qwen,"4,314",0.178066,0.608243,0.432551,0.236994,0.734223,0.182222,0.338843,0.877643,—
8,Hospital Removal,Full Retraining,"4,314",0.183297,0.561090,0.354245,0.193182,0.739540,0.226164,0.168595,0.953584,487.640731
9,Retention Expiry,Scenario-Matched Original Qwen,"6,262",0.173793,0.594546,0.398914,0.225670,0.743385,0.185185,0.288809,0.900283,—


## 7.3 Forgetting Comparison

In [273]:
forgetting_results = all_results.loc[
    all_results["Method"].isin(METHOD_LABELS.values()),
    ["Scenario", "Method", "KS Statistic ↓", "KS p-value ↑"],
].copy()
display(forgetting_results if not forgetting_results.empty else "No completed approximate forgetting results.")

'No completed approximate forgetting results.'

## 7.4 Utility Deltas

In [274]:
deltas_original = pd.concat(
    [frame for frame in [rsft_deltas, ga_deltas, gd_deltas] if not frame.empty],
    ignore_index=True,
) if any(not frame.empty for frame in [rsft_deltas, ga_deltas, gd_deltas]) else pd.DataFrame()

original_columns = ["Scenario", "Method"] + [c for c in deltas_original.columns if "vs Original" in c]
full_columns = ["Scenario", "Method"] + [c for c in deltas_original.columns if "vs Full Retraining" in c]
deltas_vs_original = deltas_original[original_columns] if not deltas_original.empty else pd.DataFrame()
deltas_vs_full = deltas_original[full_columns] if not deltas_original.empty else pd.DataFrame()
display(deltas_vs_original if not deltas_vs_original.empty else "No approximate deltas yet.")
display(deltas_vs_full if not deltas_vs_full.empty else "No approximate deltas yet.")

'No approximate deltas yet.'

'No approximate deltas yet.'

## 7.5 Computational Cost

In [275]:
runtime_rows = []
for scenario in SCENARIOS:
    reference = full_retraining[scenario]["complete"]
    runtime_rows.append({"Scenario": SCENARIO_LABELS[scenario], "Method": "Full Retraining",
        "Training Time (minutes)": reference["training_seconds"] / 60,
        "Peak GPU Memory (GiB)": reference.get("peak_gpu_memory_gib", np.nan)})
    for method, label in METHOD_LABELS.items():
        saved = load_method_results(method)
        if scenario in saved:
            complete = saved[scenario]["complete"]
            runtime_rows.append({"Scenario": SCENARIO_LABELS[scenario], "Method": label,
                "Training Time (minutes)": complete["training_seconds"] / 60,
                "Peak GPU Memory (GiB)": complete.get("peak_gpu_memory_gib", np.nan)})
runtime_summary = pd.DataFrame(runtime_rows)
display(runtime_summary.round(3))

,Scenario,Method,Training Time (minutes),Peak GPU Memory (GiB)
0,Recipient Withdrawal,Full Retraining,174.945,10.164
1,Donor Withdrawal,Full Retraining,463.881,10.164
2,Invalid Consent,Full Retraining,517.404,10.164
3,Hospital Removal,Full Retraining,487.641,10.164
4,Retention Expiry,Full Retraining,503.377,10.164


## 7.6 Utility vs Forgetting

The desirable region is top-left: high retained-test PR-AUC and low KS distance.

In [276]:
completed_all_methods = len(forgetting_results) == len(SCENARIOS) * len(METHOD_LABELS)
if completed_all_methods:
    import matplotlib.pyplot as plt
    plot_data = all_results.loc[all_results["Method"].isin(METHOD_LABELS.values())]
    fig, ax = plt.subplots(figsize=(8, 5))
    for method, group in plot_data.groupby("Method"):
        ax.scatter(group["KS Statistic ↓"], group["PR-AUC ↑"], label=method)
    ax.set(xlabel="KS Statistic", ylabel="Retained-Test PR-AUC",
           title="Qwen Unlearning: Utility vs Forgetting")
    ax.legend()
    ax.grid(alpha=0.25)
    figure_root = UNLEARNING_ROOT / "figures"
    figure_root.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(figure_root / "utility_vs_forgetting.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("Plot deferred until all 15 approximate scenario results are complete.")

Plot deferred until all 15 approximate scenario results are complete.


## 7.7 Forgetting by Scenario

In [277]:
if completed_all_methods:
    plot_data = forgetting_results.pivot(
        index="Scenario", columns="Method", values="KS Statistic ↓",
    )
    ax = plot_data.plot(kind="bar", figsize=(10, 5))
    ax.set(ylabel="KS Statistic", title="Qwen Forgetting Distance from Full Retraining")
    ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", alpha=0.25)
    figure_root = UNLEARNING_ROOT / "figures"
    figure_root.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(figure_root / "forgetting_ks_by_method.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("Plot deferred until all 15 approximate scenario results are complete.")

Plot deferred until all 15 approximate scenario results are complete.


## 7.8 Save Combined Summaries

In [278]:
if completed_all_methods:
    # Raw scenario outputs are never overwritten; these are derived summaries only.
    UNLEARNING_ROOT.mkdir(parents=True, exist_ok=True)
    figure_root = UNLEARNING_ROOT / "figures"
    figure_root.mkdir(parents=True, exist_ok=True)
    all_results.to_csv(UNLEARNING_ROOT / "qwen_unlearning_all_results.csv", index=False)
    deltas_vs_original.to_csv(
        UNLEARNING_ROOT / "qwen_unlearning_utility_deltas_vs_original.csv", index=False,
    )
    deltas_vs_full.to_csv(
        UNLEARNING_ROOT / "qwen_unlearning_utility_deltas_vs_full_retraining.csv", index=False,
    )
    forgetting_results.to_csv(
        UNLEARNING_ROOT / "qwen_unlearning_forgetting_results.csv", index=False,
    )
    runtime_summary.to_csv(
        UNLEARNING_ROOT / "qwen_unlearning_runtime_summary.csv", index=False,
    )
    print("Combined summaries saved.")
else:
    print("Combined summaries are not written until all approximate runs are complete.")

Combined summaries are not written until all approximate runs are complete.


## 7.9 Overall Findings

Findings will be added only after all three methods have completed all five scenarios. The final interpretation will keep utility preservation, forgetting distance, scenario variation, and computational cost as separate evidence.

In [279]:
FROZEN_STATE_AFTER = frozen_file_state()
assert FROZEN_STATE_AFTER == FROZEN_STATE_BEFORE, (
    "A frozen Original Qwen or Full Retraining artefact changed unexpectedly."
)
print("Frozen Baseline and Full Retraining artefacts are unchanged.")
print("All method-specific training switches remain disabled:", not METHOD_SWITCH_ENABLED)

Frozen Baseline and Full Retraining artefacts are unchanged.
All method-specific training switches remain disabled: True
